# JWST Multiphase Science Pipeline Master Dashboard
**Target Science:** IC 348 Brown Dwarf & Planetary Mass Survey  
**PI Designation:** Oleg G. Kapranov  
**Instrument System:** JWST NIRCam Broad & Narrow Filter Configurations 

This master workspace maps, triggers, and evaluates Level 1 (Detector) and Level 2 (Calibrated Image) sequences across five concurrent NIRCam setups according to the documented framework logs.

## Step 1: Environment Integration, Token Verification & Routing Paths

In [1]:
import os
import sys
import glob
import pandas as pd
from astropy.table import Table

# 1. Secure Core Pipeline Environment Variables
os.environ["MAST_API_TOKEN"] = "0bdabb80135e42a8a77929ae012b343a"  # Replace with active token
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"
os.environ["CRDS_PATH"] = os.path.expanduser("~/crds_cache")

print("=== JWST Science Pipeline Environment Anchored ===")
print(f"CRDS Cache Directory Location: {os.environ['CRDS_PATH']}")

=== JWST Science Pipeline Environment Anchored ===
CRDS Cache Directory Location: /home/kapranov/crds_cache


## Step 2: Data Discovery (Pulling Down Pool Catalogs & Association Definitions)

In [ ]:
import os
import sys

# Define your 5 clean target filter directories explicitly
target_filter_paths = [
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f162m/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f164n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f322w2-f323n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f466n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f470n/mastDownload/JWST"
]
#
# Ensure the required subdirectories exist before running metadata search sync scripts
for path in target_filter_paths:
    os.makedirs(path, exist_ok=True)

search_scripts = [
    "./search_jw01257-o003_20260714t164831.py",
    "./search_jw01257-o003_20260714t164831_00002.py",
    "./search_jw01257-o003_20260714t164831_00005.py",
    "./search_jw01257-o003_20260714t164831_00007.py",
    "./search_jw01257-o003_20260714t164831_00008.py",
    "./search_jw01257-o003_20260714t164831_00009.py"
]

print("==========================================")
print("RUNNING STEP 2: METADATA SPACE DISCOVERY")
print("==========================================")

for script in search_scripts:
    if os.path.exists(script):
        print(f"Executing discovery sync: {script}")
        try:
            # Pass globals() to make sure os and sys cross over cleanly into the scripts
            exec(open(script).read(), globals())
            print(f" -> SUCCESS: {script} synchronized cleanly.")
        except Exception as e:
            print(f" -> ERROR executing {script}: {e}")
    else:
        print(f"Skipping (not found): {script}")

print("==========================================")
print("Step 2 Database Space Discovery Finalized Successfully.")
print("==========================================")

## Step 3: Archive Retrieval (Executing Association Space Downloads from MAST Archive)

In [ ]:
import os
import sys

# Pass ONLY the clean filenames. download_asn_members.py automatically appends its internal 'mast_jwst_root' path.
download_matrix = [
    {"asn_name": "jw01257-o003_20260714t164831_image3_00002_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f162m/"},
    {"asn_name": "jw01257-o003_20260714t164831_image3_00009_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f164n/"},
    {"asn_name": "jw01257-o003_20260714t164831_image3_00005_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f322w2-f323n/"},
    {"asn_name": "jw01257-o003_20260714t164831_image3_00008_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f466n/"},
    {"asn_name": "jw01257-o003_20260714t164831_image3_00007_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f470n/"}
]

if os.path.exists("./download_asn_members.py"):
    print("--- Starting Clean-Aligned Multi-Filter Download Matrix ---")

    for job in download_matrix:
        print(f"\nProcessing Group: {job['asn_name']}")
        print(f" -> Target Directory: {job['dest_dir']}")

        # We pass ONLY the filename as parameter 1. The utility script prepends its internal path cleanly.
        sys.argv = ["./download_asn_members.py", job["asn_name"], "1257", job["dest_dir"]]

        # Execute the download routine cleanly
        exec(open("./download_asn_members.py").read(), globals())
else:
    print("ERROR: download_asn_members.py utility script missing in root layout.")


## Step 4: Health-Check Logistics (Physical Footprint Verification Metrics)

In [ ]:
import os
import glob

target_filter_paths = [
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f162m/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f164n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f322w2-f323n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f466n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f470n/mastDownload/JWST"
]

total_recovered = 0
print("==========================================")
print("FILTER WORKSPACE RETRIEVAL VERIFICATION")
print("==========================================")

for path in target_filter_paths:
    if os.path.exists(path):
        files_in_filter = glob.glob(os.path.join(path, "jw*/*_uncal.fits"))
        folder_name = path.split('/')[2]
        print(f" -> Filter Workspace: {folder_name:<50} | Exposures Found: {len(files_in_filter)}")
        total_recovered += len(files_in_filter)

print("==========================================")
print(f"Total clean exposures matched for processing: {total_recovered} files")
print("==========================================")

## Step 5: Execute Complete Calibration Matrix (Detector1Pipeline & Image2Pipeline Workflow)

In [ ]:
# Execute Stage 1 and Stage 2 Pipelines sequentially over the data pools
pipeline_scripts = ["./run_stage_1_pipeline.py", "./run_stage_2_pipeline.py"]

for pipeline_run in pipeline_scripts:
    if os.path.exists(pipeline_run):
        print(f"\n--- Invoking Calibration Sequence: {pipeline_run} ---")
        exec(open(pipeline_run).read())
    else:
        print(f"Warning: Pipeline controller script {pipeline_run} not located.")

print("\nProcessing complete! All exposures successfully converted to standard *_cal.fits structural formats.")


--- Invoking Calibration Sequence: ./run_stage_1_pipeline.py ---
Located 32 raw files for structured pipeline processing.

--- Invoking Calibration Sequence: ./run_stage_2_pipeline.py ---
Located 32 raw files for pipeline processing inside ./processed_stage1_raw/

Processing Target: jw01257003001_02109_00002_nrcb1_uncal.fits
--- Running Stage 1: Detector Processing ---



--- Running Stage 2: Image Processing for jw01257003001_02109_00002_nrcb1_rate.fits ---



Processing Target: jw01257003001_02109_00003_nrca2_uncal.fits
--- Running Stage 1: Detector Processing ---



--- Running Stage 2: Image Processing for jw01257003001_02109_00003_nrca2_rate.fits ---



Processing Target: jw01257003001_02109_00002_nrcb4_uncal.fits
--- Running Stage 1: Detector Processing ---



--- Running Stage 2: Image Processing for jw01257003001_02109_00002_nrcb4_rate.fits ---


<div style='padding: 15px; border: 2px solid #D9534F; border-radius: 5px; background-color: #FDF2F2; color: #B94A48;'>
    <h3 style='margin-top: 0; color: #D9534F; font-weight: bold;'>⚠️ STAGE 5: CRITICAL STORAGE & DISK SPACE WARNING</h3>
    <p style='font-size: 13px; line-height: 1.5;'>
        <strong>ATTENTION OPERATOR:</strong> JWST raw exposure calibrations require massive array allocations. To prevent local system drive saturation, <strong>DO NOT</strong> dump large batches of files into the ingestion path simultaneously.
    </p>
    <p style='font-size: 13px; line-height: 1.5;'>
        Process exposures sequentially by dropping <strong>ONLY ONE raw file at a time</strong> into the target directory path:<br>
        <code>./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f164n/</code>
    </p>
    <p style='font-size: 13px; line-height: 1.5; margin-bottom: 0;'>
        Failure to partition inputs will expand the local cache directory footprint beyond system limits (<strong>~50 to 100 GB allocation spike</strong>), resulting in a hard kernel panic crash.
    </p>
</div>